# Dashboard mart from Excel INN list (March)

Notebook builds final dashboard dataset for all clients whose INN exists in Excel for March.

Logic baseline: `final_attrs_single_inn_7708044880.ipynb`, scaled to full Excel INN perimeter with fallback key (`agr_id` or `INN + contract_number`).

Main constraints:
- SA contracts active on period;
- transaction perimeter from DAG: `c_trx_class='SA'`, `c_trx_type='S01'`, `cf_trx_stat<>'R'`, `c_nter is not null`, `ods_deleted_flg<>'1'`;
- FIID filter: `coalesce(fa.c_fiid_grp, 'UNKNOWN') = 'RSHB'`;
- no hard block if `agr_id` is missing.

In [ ]:
import gc
import re
import warnings
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

# Suppress noisy non-critical pandas warnings from connector internals.
warnings.filterwarnings(
    'ignore',
    category=FutureWarning,
    message='The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*'
)

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Parameters
report_month = '2026-03-01'
month_start = pd.to_datetime(report_month).strftime('%Y-%m-%d')
month_end = (pd.to_datetime(report_month) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

excel_path = '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx'
excel_col_inn = 'ИНН'

sa_type = 'SA'
aur_rate = 1926.0
mem_limit = '10g'

output_dir = '/home/jovyan/documents/Equaring/output'

print('report_month =', report_month)
print('month_start =', month_start)
print('month_end =', month_end)
print('excel_path =', excel_path)
print('sa_type =', sa_type)
print('mem_limit =', mem_limit)
print('output_dir =', output_dir)

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    if not s:
        return None
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s
    return s


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    s = re.sub(r'\s+', '', s)
    return s if s else None


def to_sql_in_list(values):
    out = []
    for v in values:
        s = str(v).replace("'", "''")
        out.append("'" + s + "'")
    return ', '.join(out)


def chunk_list(values, chunk_size):
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


def fetch_in_chunks(imp, values, chunk_size, sql_builder):
    out_df = pd.DataFrame()
    for part in chunk_list(values, chunk_size):
        in_list = to_sql_in_list(part)
        if not in_list:
            continue
        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            d = imp.fetch(sql_builder(in_list))
        if len(d) == 0:
            continue
        out_df = d if len(out_df) == 0 else pd.concat([out_df, d], ignore_index=True)
        del d
        gc.collect()
    return out_df

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods.scd1_z_r2_ip_merchants')
    imp.execute('invalidate metadata ods.scd1_z_cl_corp')
    imp.execute('invalidate metadata ocrm_ul.s_org_ext')
    imp.execute('invalidate metadata cdiul.ext_id_org')

print('Impala initialized (fast mode)')

## 1) Excel INN perimeter (normalized)

In [ ]:
excel_raw = pd.read_excel(excel_path)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

if excel_col_inn not in excel_raw.columns:
    raise ValueError(f'Missing Excel INN column: {excel_col_inn}')

excel_inn_df = excel_raw[[excel_col_inn]].copy()
excel_inn_df.columns = ['inn_raw']
excel_inn_df['inn'] = excel_inn_df['inn_raw'].apply(normalize_numeric_str)
excel_inn_df = excel_inn_df[(excel_inn_df['inn'].notna()) & (excel_inn_df['inn'] != '')].copy()
excel_inn_df = excel_inn_df[['inn']].drop_duplicates().sort_values('inn').reset_index(drop=True)

excel_inn_list = excel_inn_df['inn'].tolist()
print('excel_inn_cnt =', len(excel_inn_list))

## 2) Lake INN filter: active SA agreements for INN from Excel

In [ ]:
def build_sql_lake_inn(in_list):
    return f"""
    select distinct
        regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
    where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') in ({in_list})
      and upper(trim(cast(a.acq_class as string))) = '{sa_type}'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
    """

lake_inn_df = fetch_in_chunks(imp, excel_inn_list, 700, build_sql_lake_inn)
if len(lake_inn_df):
    lake_inn_df['inn'] = lake_inn_df['inn'].apply(normalize_numeric_str)
lake_inn_set = set(lake_inn_df['inn'].dropna().tolist()) if len(lake_inn_df) else set()

excel_src_df = excel_raw.copy()
excel_src_df['inn_norm'] = excel_src_df[excel_col_inn].apply(normalize_numeric_str)
rows_before = len(excel_src_df)
inn_before = excel_src_df['inn_norm'].dropna().nunique()

base_df = excel_src_df[excel_src_df['inn_norm'].isin(lake_inn_set)].copy()
rows_after = len(base_df)
inn_after = base_df['inn_norm'].dropna().nunique()

print('lake_inn_cnt =', len(lake_inn_set))
print('rows_before =', rows_before)
print('rows_after =', rows_after)
print('unique_inn_before =', inn_before)
print('unique_inn_after =', inn_after)

display(base_df.head(30))

## 3) Lake attributes only: `OGRN` by INN

In [ ]:
if len(lake_inn_set) == 0:
    ogrn_df = pd.DataFrame(columns=['inn_norm', 'ogrn'])
else:
    def build_sql_ogrn(in_list):
        return f"""
        with b as (
          select distinct
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_norm,
            cast(a.abs_agr_id as string) as agr_id
          from ods_alpha.scd1_agreements a
          join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
          where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') in ({in_list})
            and upper(trim(cast(a.acq_class as string))) = '{sa_type}'
            and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
            and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
            and coalesce(a.ods_deleted_flg, '0') <> '1'
            and coalesce(c.ods_deleted_flg, '0') <> '1'
            and a.abs_agr_id is not null
        )
        select
          b.inn_norm,
          cast(corp.c_register_gos_reg_num_rec as string) as ogrn
        from b
        left join ods.scd1_z_r2_ip_merchants m on cast(m.id as string) = b.agr_id
        left join ods.scd1_z_cl_corp corp on corp.id = m.c_cl_org
        """

    ogrn_raw_df = fetch_in_chunks(imp, sorted(list(lake_inn_set)), 700, build_sql_ogrn)
    if len(ogrn_raw_df):
        ogrn_raw_df['inn_norm'] = ogrn_raw_df['inn_norm'].apply(normalize_numeric_str)
        ogrn_raw_df['ogrn'] = ogrn_raw_df['ogrn'].astype(str).str.strip()
        ogrn_raw_df.loc[ogrn_raw_df['ogrn'].isin(['', 'None', 'nan']), 'ogrn'] = None
        ogrn_df = (
            ogrn_raw_df.dropna(subset=['inn_norm'])
            .sort_values(['inn_norm', 'ogrn'])
            .drop_duplicates('inn_norm', keep='first')[['inn_norm', 'ogrn']]
            .reset_index(drop=True)
        )
    else:
        ogrn_df = pd.DataFrame(columns=['inn_norm', 'ogrn'])

display(ogrn_df.head(30))
print('ogrn_rows =', len(ogrn_df))

## 4) OCRM mapping (`cdi_id`, `ssp_ocrm`) by INN

In [ ]:
def build_sql_ocrm(in_list):
    return f"""
    with t as (
      select
        cast(row_id as string) as row_id,
        regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') as inn_norm,
        case
          when upper(trim(cast(x_area_resp as string))) like 'ДММБ%' or upper(trim(cast(x_area_resp as string))) like 'ДМСБ (МИ%)' then 'ДМ'
          when upper(trim(cast(x_area_resp as string))) like 'ДМСБ (МА%)' then 'ДМСБ'
          when upper(trim(cast(x_area_resp as string))) like 'ДМСБ (СР%)' or upper(trim(cast(x_area_resp as string))) like 'ДСБ%' then 'ДМСБ'
          when upper(trim(cast(x_area_resp as string))) like 'ДКБ%' then 'ДКБ'
          when upper(trim(cast(x_area_resp as string))) like 'ДРПА%' then 'ДРПА'
          when lower(trim(cast(x_area_resp as string))) like 'не под%' then 'Не подлежит сегментации'
          when nullif(trim(cast(x_area_resp as string)), '') is null then null
          else trim(cast(x_area_resp as string))
        end as ssp_ocrm,
        created,
        row_number() over (
          partition by regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '')
          order by created desc, row_id desc
        ) as rn
      from ocrm_ul.s_org_ext
      where x_removed_flg = 'N'
        and x_duplicate_flg = 'N'
        and regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') in ({in_list})
    ),
    m as (
      select distinct
        cast(party_id as string) as cdi_id,
        cast(cmo_ext_party_source_id as string) as row_id
      from cdiul.ext_id_org
      where cmo_ext_source_system like 'OCRM%'
    )
    select
      t.inn_norm,
      m.cdi_id,
      t.ssp_ocrm
    from t
    left join m on m.row_id = t.row_id
    where t.rn = 1
    """

ocrm_df = fetch_in_chunks(imp, sorted(list(lake_inn_set)), 700, build_sql_ocrm) if len(lake_inn_set) else pd.DataFrame(columns=['inn_norm','cdi_id','ssp_ocrm'])
if len(ocrm_df):
    ocrm_df['inn_norm'] = ocrm_df['inn_norm'].apply(normalize_numeric_str)

display(ocrm_df.head(30))
print('ocrm_rows =', len(ocrm_df))

## 5) Transaction metrics by `n_agr` with DAG perimeter + `FIID=RSHB`

In [ ]:
if len(base_df) == 0:
    trx_df = pd.DataFrame(columns=['n_agr','trx_cnt','trx_sum','commission_from_ops','int_component'])
else:
    n_agrs = sorted(base_df['n_agr'].dropna().astype(str).unique().tolist())
    trx_df = pd.DataFrame(columns=['n_agr','trx_cnt','trx_sum','commission_from_ops','int_component'])

    for part in chunk_list(n_agrs, 400):
        agr_in = to_sql_in_list(part)
        sql_trx = f"""
        with trx_base as (
          select cast(t.n_trx as string) as n_trx,
                 cast(t.n_amt_src as double) as n_amt_src
          from ods_alpha.scd1_trx t
          left join ods_alpha.scd1_base24_fiids fa on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
          where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
            and t.c_nter is not null
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and t.c_trx_class = 'SA'
            and t.c_trx_type = 'S01'
            and coalesce(t.cf_trx_stat, '') <> 'R'
            and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
        ),
        ta as (
          select cast(n_trx as string) as n_trx,
                 cast(n_agr as string) as n_agr,
                 coalesce(cast(n_amt_tax as double), 0.0) as n_amt_tax
          from ods_alpha.scd1_trx_acq
          where cast(n_agr as string) in ({agr_in})
        ),
        tj as (
          select ta.n_agr, ta.n_trx, tb.n_amt_src, ta.n_amt_tax
          from ta join trx_base tb on tb.n_trx = ta.n_trx
        )
        select
          tj.n_agr,
          count(distinct tj.n_trx) as trx_cnt,
          sum(tj.n_amt_src) as trx_sum,
          sum(tj.n_amt_tax) as commission_from_ops,
          sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as int_component
        from tj
        left join ods_alpha.scd1_trx_int i on cast(i.n_trx as string) = tj.n_trx
        group by tj.n_agr
        """

        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            d = imp.fetch(sql_trx)
        if len(d):
            trx_df = d if len(trx_df) == 0 else pd.concat([trx_df, d], ignore_index=True)
        del d
        gc.collect()

display(trx_df.head(30))
print('trx_rows =', len(trx_df))

## 6) OCRM mapping (`cdi_id`, `ssp_ocrm`) by Excel INN

In [ ]:
def build_sql_ocrm(in_list):
    return f"""
    with t as (
      select
        cast(row_id as string) as row_id,
        regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') as inn,
        case
          when upper(trim(cast(x_area_resp as string))) like 'ДММБ%' or upper(trim(cast(x_area_resp as string))) like 'ДМСБ (МИ%)' then 'ДМ'
          when upper(trim(cast(x_area_resp as string))) like 'ДМСБ (МА%)' then 'ДМСБ'
          when upper(trim(cast(x_area_resp as string))) like 'ДМСБ (СР%)' or upper(trim(cast(x_area_resp as string))) like 'ДСБ%' then 'ДМСБ'
          when upper(trim(cast(x_area_resp as string))) like 'ДКБ%' then 'ДКБ'
          when upper(trim(cast(x_area_resp as string))) like 'ДРПА%' then 'ДРПА'
          when lower(trim(cast(x_area_resp as string))) like 'не под%' then 'Не подлежит сегментации'
          when nullif(trim(cast(x_area_resp as string)), '') is null then null
          else trim(cast(x_area_resp as string))
        end as ssp_ocrm,
        created,
        row_number() over (
          partition by regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '')
          order by created desc, row_id desc
        ) as rn
      from ocrm_ul.s_org_ext
      where x_removed_flg = 'N'
        and x_duplicate_flg = 'N'
        and regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') in ({in_list})
    ),
    m as (
      select distinct
        cast(party_id as string) as cdi_id,
        cast(cmo_ext_party_source_id as string) as row_id
      from cdiul.ext_id_org
      where cmo_ext_source_system like 'OCRM%'
    )
    select
      t.inn,
      t.row_id,
      m.cdi_id,
      t.ssp_ocrm
    from t
    left join m on m.row_id = t.row_id
    where t.rn = 1
    """

ocrm_df = fetch_in_chunks(imp, excel_inn_list, 700, build_sql_ocrm)
if len(ocrm_df):
    ocrm_df['inn'] = ocrm_df['inn'].apply(normalize_numeric_str)

display(ocrm_df.head(30))
print('ocrm_rows =', len(ocrm_df))

## 7) Final dataset and KPI calculations

In [ ]:
attrs_df = base_df.copy()

if len(r2_df):
    r2_tier1 = r2_df[r2_df['agr_id'].notna()].copy()
    if len(r2_tier1):
        attrs_df = attrs_df.merge(
            r2_tier1[['agr_id','ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly']].drop_duplicates('agr_id'),
            on='agr_id',
            how='left'
        )

if len(r2_df):
    r2_tier2 = r2_df[r2_df['agr_id'].isna()].copy()
    if len(r2_tier2):
        attrs_df = attrs_df.merge(
            r2_tier2[['inn','contract_number','ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly']].drop_duplicates(['inn','contract_number']),
            on=['inn','contract_number'],
            how='left',
            suffixes=('', '_tier2')
        )
        for c in ['ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly']:
            c2 = f"{c}_tier2"
            if c2 in attrs_df.columns:
                attrs_df[c] = attrs_df[c].where(attrs_df[c].notna(), attrs_df[c2]) if c in attrs_df.columns else attrs_df[c2]
                attrs_df.drop(columns=[c2], inplace=True)

if len(cmp_df):
    cmp_df = cmp_df[['n_cmp_client','retl_cnt','term_cnt','mcc_any','amortization']].drop_duplicates('n_cmp_client')
    attrs_df = attrs_df.merge(cmp_df, on='n_cmp_client', how='left')
if len(trx_df):
    trx_df = trx_df[['n_agr','trx_cnt','trx_sum','commission_from_ops','int_component']].drop_duplicates('n_agr')
    attrs_df = attrs_df.merge(trx_df, on='n_agr', how='left')
if len(ocrm_df):
    ocrm_join_df = ocrm_df[['inn', 'row_id', 'cdi_id', 'ssp_ocrm']].drop_duplicates().rename(columns={'row_id': 'ocrm_row_id'})
    attrs_df = attrs_df.merge(ocrm_join_df, on='inn', how='left')

for c in ['retl_cnt','term_cnt','trx_cnt','trx_sum','commission_from_ops','commission_monthly','int_component','amortization']:
    if c in attrs_df.columns:
        attrs_df[c] = pd.to_numeric(attrs_df[c], errors='coerce').fillna(0)

attrs_df['key_tier'] = attrs_df['agr_id'].apply(
    lambda x: 'tier1_agr_id' if pd.notna(x) and str(x).strip() != '' else 'tier2_inn_contract'
)
attrs_df['dq_block_reason'] = attrs_df.apply(
    lambda r: None if (pd.notna(r.get('contract_number')) and str(r.get('contract_number')).strip() != '') else 'missing_contract_number',
    axis=1
)

attrs_df['commission_total'] = attrs_df.get('commission_from_ops', 0) + attrs_df.get('commission_monthly', 0)
attrs_df['aur'] = attrs_df.get('term_cnt', 0) * aur_rate
attrs_df['nod'] = attrs_df.get('commission_total', 0) - attrs_df.get('int_component', 0)
attrs_df['fin_result'] = attrs_df.get('nod', 0) - attrs_df.get('aur', 0) - attrs_df.get('amortization', 0)
attrs_df['report_month'] = report_month

final_columns = [
    'report_month',
    'inn', 'company_name', 'n_cmp_client', 'cdi_id', 'ocrm_row_id', 'ssp_ocrm',
    'contract_type', 'key_tier', 'dq_block_reason',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name', 'ogrn', 'mcc_any',
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'nod', 'fin_result'
]
final_columns = [c for c in final_columns if c in attrs_df.columns]
final_df = attrs_df[final_columns].copy()

del attrs_df
gc.collect()

display(final_df.head(30))
print('final_rows =', len(final_df))
print('final_unique_inn =', final_df['inn'].nunique() if len(final_df) else 0)
print('tier_counts =')
display(final_df['key_tier'].value_counts(dropna=False).reset_index(name='rows').rename(columns={'index': 'key_tier'}))

## 8) Reconciliation with Excel INN perimeter + output export

In [ ]:
excel_set = set(excel_inn_list)
final_set = set(final_df['inn'].dropna().astype(str).tolist()) if len(final_df) else set()

recon_df = pd.DataFrame([
    {'metric': 'excel_unique_inn', 'value': len(excel_set)},
    {'metric': 'final_unique_inn', 'value': len(final_set)},
    {'metric': 'inn_in_both', 'value': len(excel_set & final_set)},
    {'metric': 'inn_only_in_excel', 'value': len(excel_set - final_set)},
    {'metric': 'inn_only_in_final', 'value': len(final_set - excel_set)},
])
display(recon_df)

pd.DataFrame({'inn': sorted(excel_set - final_set)}).head(30)

out_dir = Path(output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

detail_parquet = out_dir / f"dashboard_mart_detail_{report_month}.parquet"
detail_csv = out_dir / f"dashboard_mart_detail_{report_month}.csv"
key_parquet = out_dir / f"dashboard_mart_fallback_key_{report_month}.parquet"
key_csv = out_dir / f"dashboard_mart_fallback_key_{report_month}.csv"

final_df.to_parquet(detail_parquet, index=False)
final_df.to_csv(detail_csv, index=False)

agg_dict = {
    'company_name': 'max',
    'n_cmp_client': 'max',
    'cdi_id': 'max',
    'ssp_ocrm': 'max',
    'trx_cnt': 'sum',
    'trx_sum': 'sum',
    'commission_from_ops': 'sum',
    'commission_monthly': 'max',
    'int_component': 'sum',
    'commission_total': 'sum',
    'term_cnt': 'max',
    'retl_cnt': 'max',
    'aur': 'sum',
    'amortization': 'sum',
    'nod': 'sum',
    'fin_result': 'sum',
}
agg_dict = {k: v for k, v in agg_dict.items() if k in final_df.columns}

key_df = final_df.copy()
key_df['fallback_key'] = key_df.apply(
    lambda r: str(r['agr_id']) if pd.notna(r.get('agr_id')) and str(r.get('agr_id')).strip() != '' else f"{r.get('inn')}|{r.get('contract_number')}",
    axis=1
)
key_group_cols = [c for c in ['report_month', 'inn', 'contract_number', 'agr_id', 'fallback_key'] if c in key_df.columns]
key_df = key_df.groupby(key_group_cols, as_index=False).agg(agg_dict)

key_df.to_parquet(key_parquet, index=False)
key_df.to_csv(key_csv, index=False)

print('Saved:')
print(detail_parquet)
print(detail_csv)
print(key_parquet)
print(key_csv)
print('key_rows =', len(key_df))
display(key_df.head(30))

del key_df
gc.collect()